In [9]:
!pip install -r ../requirements.txt

In [11]:
import os
import re
import json
import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import xgboost as xgb
from joblib import Parallel, delayed
from sklearn.metrics import f1_score, classification_report
from sklearn.metrics.pairwise import paired_cosine_distances

import warnings
warnings.filterwarnings('ignore')

# 1. Load Data
DATA_DIR  = "./Data"
MODEL_DIR = "./Models_ML"

dev_df = pd.read_csv(os.path.join(DATA_DIR, "dev.csv"))
print(f"Dev: {len(dev_df):,}")

# 2. Load Models & Vectorizers
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(os.path.join(MODEL_DIR, "xgb_model.json"))
lgb_model = joblib.load(os.path.join(MODEL_DIR, "lgb_model.pkl"))

v_w  = joblib.load(os.path.join(MODEL_DIR, "tfidf_word.pkl"))
v_c  = joblib.load(os.path.join(MODEL_DIR, "tfidf_char.pkl"))
v_fw = joblib.load(os.path.join(MODEL_DIR, "tfidf_fw.pkl"))

with open(os.path.join(MODEL_DIR, "ensemble_config.json")) as f:
    cfg = json.load(f)

xgb_weight = cfg['xgb_weight']
lgb_weight  = cfg['lgb_weight']
threshold   = cfg['threshold']
print(f"Ensemble — XGB: {xgb_weight} | LGB: {lgb_weight} | Threshold: {threshold}")

# 3. Preprocessing
def normalize_text(text):
    text = str(text)
    text = re.sub(r'[\w.+-]+@[\w.]+\.[a-z]{2,}', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}', ' ', text)
    text = re.sub(r'\d{1,2}:\d{2}(:\d{2})?', ' ', text)
    text = re.sub(r'\(?\d{3}\)?[-.\s]\d{3}[-.\s]\d{4}', ' ', text)
    text = re.sub(r'(.)\1{3,}', r'\1\1', text)
    text = re.sub(r'([!?.]){2,}', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def process_text_stylometric(text):
    text = str(text)
    words = text.split()
    ln = len(words)
    if ln < 5:
        return np.zeros(24, dtype='float32')
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    word_lens = [len(w) for w in words]
    sent_lens = [len(s.split()) for s in sentences] if sentences else [ln]
    return np.array([
        ln, np.mean(word_lens), np.mean(sent_lens), len(set(words)) / ln,
        np.std(word_lens), np.std(sent_lens), np.max(word_lens), np.median(sent_lens),
        text.count(',') / ln, text.count('.') / ln, text.count('!') / ln, text.count('?') / ln,
        text.count(';') / ln, text.count(':') / ln, text.count('"') / ln, text.count("'") / ln,
        sum(w.isupper() for w in words) / ln, sum(len(w) > 6 for w in words) / ln,
        sum(len(w) <= 3 for w in words) / ln, sum(w[0].isupper() for w in words if w) / ln,
        sum(w.endswith('ly') for w in words) / ln, sum(w.endswith(('tion', 'sion')) for w in words) / ln,
        sum(w.endswith('ing') for w in words) / ln, sum(w.endswith('ed') for w in words) / ln,
    ], dtype='float32')

def fast_feature_extraction(df):
    texts_1 = df['text_1_clean'].values
    texts_2 = df['text_2_clean'].values
    all_texts = np.concatenate([texts_1, texts_2])
    mid = len(df)
    style_results = Parallel(n_jobs=-1)(delayed(process_text_stylometric)(t) for t in all_texts)
    s1 = np.array(style_results[:mid])
    s2 = np.array(style_results[mid:])
    diff  = np.abs(s1 - s2)
    ratio = np.minimum(s1, s2) / (np.maximum(s1, s2) + 1e-9)
    mean  = (s1 + s2) / 2
    return sp.csr_matrix(np.hstack([diff, ratio, mean]).astype('float32'))

def build_matrix(df, s):
    w1  = v_w.transform(df['text_1_clean'])
    c1  = v_c.transform(df['text_1_clean'])
    fw1 = v_fw.transform(df['text_1_clean'])
    w2  = v_w.transform(df['text_2_clean'])
    c2  = v_c.transform(df['text_2_clean'])
    fw2 = v_fw.transform(df['text_2_clean'])
    x1 = sp.hstack([w1, c1])
    x2 = sp.hstack([w2, c2])
    cos    = paired_cosine_distances(x1,  x2).reshape(-1, 1).astype('float32')
    fw_cos = paired_cosine_distances(fw1, fw2).reshape(-1, 1).astype('float32')
    return sp.hstack([abs(x1-x2), x1.multiply(x2), abs(fw1-fw2), fw1.multiply(fw2), s, cos, fw_cos], format='csr', dtype='float32')

# 4. Feature Extraction
dev_df['text_1_clean'] = dev_df['text_1'].apply(normalize_text)
dev_df['text_2_clean'] = dev_df['text_2'].apply(normalize_text)

dev_s        = fast_feature_extraction(dev_df)
dev_features = build_matrix(dev_df, dev_s)

# 5. Predict & Evaluate
probs_xgb = xgb_model.predict_proba(dev_features)[:, 1]
probs_lgb = lgb_model.predict_proba(dev_features)[:, 1]

final_probs = (probs_xgb * xgb_weight) + (probs_lgb * lgb_weight)
final_preds = (final_probs >= threshold).astype(int)

print(f"F1 Score : {f1_score(dev_df['label'], final_preds):.4f}")
print(f"Macro F1 Score : {f1_score(dev_df['label'], final_preds, average='macro'):.4f}")
print(classification_report(dev_df['label'], final_preds))

# 6. Sample Preview
dev_df['pred']    = final_preds
dev_df['prob']    = final_probs.round(4)
dev_df['correct'] = (dev_df['pred'] == dev_df['label'])

Dev: 5,993
Ensemble — XGB: 0.26 | LGB: 0.74 | Threshold: 0.43
F1 Score : 0.7813
Macro F1 Score : 0.7700
              precision    recall  f1-score   support

           0       0.78      0.74      0.76      2937
           1       0.76      0.80      0.78      3056

    accuracy                           0.77      5993
   macro avg       0.77      0.77      0.77      5993
weighted avg       0.77      0.77      0.77      5993



In [ ]:
# Demo Prediction
demo_df = pd.read_csv(os.path.join(DATA_DIR, "dev.csv"))
print(f"Demo: {len(demo_df):,}")

demo_df['text_1_clean'] = demo_df['text_1'].apply(normalize_text)
demo_df['text_2_clean'] = demo_df['text_2'].apply(normalize_text)

demo_s        = fast_feature_extraction(demo_df)
demo_features = build_matrix(demo_df, demo_s)

probs_xgb = xgb_model.predict_proba(demo_features)[:, 1]
probs_lgb = lgb_model.predict_proba(demo_features)[:, 1]

final_probs = (probs_xgb * xgb_weight) + (probs_lgb * lgb_weight)
final_preds = (final_probs >= threshold).astype(int)

pd.DataFrame({'prediction': final_preds}).to_csv("Group33_AV_A_Prediction.csv", index=False)
print("✅ Saved → Group33_AV_A_Prediction.csv")

Demo: 5,993
✅ Saved → GroupN_AV_A_Prediction.csv


In [ ]:
# 7. Interactive Demo
def predict_author_match(text1, text2):
    temp_df = pd.DataFrame({'text_1': [text1], 'text_2': [text2]})
    temp_df['text_1_clean'] = temp_df['text_1'].apply(normalize_text)
    temp_df['text_2_clean'] = temp_df['text_2'].apply(normalize_text)
    
    s = fast_feature_extraction(temp_df)
    features = build_matrix(temp_df, s)
    
    p_xgb = xgb_model.predict_proba(features)[:, 1]
    p_lgb = lgb_model.predict_proba(features)[:, 1]
    
    prob = (p_xgb[0] * xgb_weight) + (p_lgb[0] * lgb_weight)
    pred = int(prob >= threshold)
    
    result = "✅ [Same Author]" if pred == 1 else "❌ [Different Author]"
    
    print("\n" + "="*40)
    print(f"  PREDICTION : {result}")
    print(f"  PROBABILITY: {prob*100:.2f}%")
    print(f"  THRESHOLD  : {threshold*100:.2f}%")
    print("="*40)

# Example Usage:
text_a = "Write any text here..."
text_b = "Hello this is group33"

predict_author_match(text_a, text_b)


  PREDICTION : ✅ [Same Author]
  PROBABILITY: 64.99%
  THRESHOLD  : 43.00%
